# 🎓 Sales Forecasting ML Model Training
## Thesis Project - Banelo Forecasting System

---

### 🛡️ Database Safety:
✅ This notebook works ONLY with CSV files  
✅ NO connection to PostgreSQL database  
✅ Training happens in Google Colab (isolated)  
✅ Your Railway database is completely safe  

---

### 📋 Workflow:
1. Upload CSV files (from Google Drive)
2. Explore and analyze data
3. Train forecasting model
4. Evaluate model performance
5. Download trained model (.pkl file)
6. Integrate into your Django application

---

**Author:** Your Name  
**Date:** 2025  
**Purpose:** Thesis - Sales Forecasting System


## 📦 Step 1: Install Required Libraries

In [ ]:
# Install necessary libraries
!pip install pandas numpy scikit-learn matplotlib seaborn statsmodels -q

print("✅ Libraries installed successfully!")

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import pickle

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries imported successfully!")

## 📥 Step 2: Upload CSV Files

Upload your CSV files from Google Drive:  
- `sales_data.csv` or `combined_sales_data.csv`
- `products_data.csv` (optional)

In [ ]:
from google.colab import files

print("📥 Upload your CSV file(s):")
print("   → sales_data.csv (real data only)")
print("   → combined_sales_data.csv (real + synthetic)")
print("\n⬇️ Click 'Choose Files' below:\n")

uploaded = files.upload()

print("\n✅ Files uploaded:")
for filename in uploaded.keys():
    print(f"   • {filename}")

## 🔍 Step 3: Load and Explore Data

In [ ]:
# Load sales data
# Adjust filename if needed
SALES_FILE = 'combined_sales_data.csv'  # or 'sales_data.csv'

try:
    df_sales = pd.read_csv(SALES_FILE)
    print(f"✅ Loaded: {SALES_FILE}")
    print(f"📊 Shape: {df_sales.shape[0]:,} rows × {df_sales.shape[1]} columns\n")
except FileNotFoundError:
    print(f"❌ File not found: {SALES_FILE}")
    print("💡 Update SALES_FILE variable with your uploaded filename")

In [ ]:
# Display first few rows
print("📋 First 5 rows:")
df_sales.head()

In [ ]:
# Data info
print("ℹ️ Dataset Information:\n")
df_sales.info()

In [ ]:
# Basic statistics
print("📊 Statistical Summary:\n")
df_sales.describe()

In [ ]:
# Check data types distribution (if available)
if 'data_type' in df_sales.columns:
    print("\n🏷️ Data Type Distribution:")
    print(df_sales['data_type'].value_counts())
    print(f"\n📊 Percentage:")
    print(df_sales['data_type'].value_counts(normalize=True) * 100)

## 🧹 Step 4: Data Preprocessing

In [ ]:
# Convert date columns
df_sales['order_date'] = pd.to_datetime(df_sales['order_date'])

# Extract useful date features
df_sales['date'] = df_sales['order_date'].dt.date
df_sales['year'] = df_sales['order_date'].dt.year
df_sales['month'] = df_sales['order_date'].dt.month
df_sales['day'] = df_sales['order_date'].dt.day
df_sales['day_of_week'] = df_sales['order_date'].dt.dayofweek
df_sales['week_of_year'] = df_sales['order_date'].dt.isocalendar().week
df_sales['is_weekend'] = df_sales['day_of_week'].isin([5, 6]).astype(int)

print("✅ Date features extracted")
print(f"\n📅 Date range: {df_sales['date'].min()} to {df_sales['date'].max()}")
print(f"📆 Total days: {(df_sales['date'].max() - df_sales['date'].min()).days + 1}")

In [ ]:
# Check for missing values
print("\n🔍 Missing Values:\n")
missing = df_sales.isnull().sum()
missing = missing[missing > 0]
if len(missing) > 0:
    print(missing)
else:
    print("✅ No missing values!")

## 📊 Step 5: Exploratory Data Analysis (EDA)

In [ ]:
# Sales over time
plt.figure(figsize=(15, 5))

daily_sales = df_sales.groupby('date')['quantity'].sum()

plt.plot(daily_sales.index, daily_sales.values, linewidth=2)
plt.title('📈 Daily Sales Volume Over Time', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Total Quantity Sold', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Average daily sales: {daily_sales.mean():.1f} units")
print(f"Max daily sales: {daily_sales.max():.0f} units")
print(f"Min daily sales: {daily_sales.min():.0f} units")

In [ ]:
# Top selling products
top_products = df_sales.groupby('product_name')['quantity'].sum().sort_values(ascending=False).head(10)

plt.figure(figsize=(12, 6))
top_products.plot(kind='barh', color='skyblue')
plt.title('🏆 Top 10 Products by Total Quantity Sold', fontsize=16, fontweight='bold')
plt.xlabel('Total Quantity', fontsize=12)
plt.ylabel('Product', fontsize=12)
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

In [ ]:
# Sales by category
category_sales = df_sales.groupby('category')['quantity'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 6))
plt.pie(category_sales.values, labels=category_sales.index, autopct='%1.1f%%', startangle=90)
plt.title('📊 Sales Distribution by Category', fontsize=16, fontweight='bold')
plt.axis('equal')
plt.tight_layout()
plt.show()

In [ ]:
# Weekday vs Weekend sales
weekend_sales = df_sales.groupby('is_weekend')['quantity'].sum()

plt.figure(figsize=(8, 6))
plt.bar(['Weekday', 'Weekend'], weekend_sales.values, color=['coral', 'lightblue'])
plt.title('📅 Weekday vs Weekend Sales', fontsize=16, fontweight='bold')
plt.ylabel('Total Quantity', fontsize=12)
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print(f"Weekday sales: {weekend_sales[0]:,.0f} units")
print(f"Weekend sales: {weekend_sales[1]:,.0f} units")

## 🤖 Step 6: Prepare Data for ML Training

In [ ]:
# Aggregate daily sales by product
daily_product_sales = df_sales.groupby(['date', 'product_name']).agg({
    'quantity': 'sum',
    'total': 'sum',
    'category': 'first',
    'day_of_week': 'first',
    'is_weekend': 'first',
    'week_of_year': 'first',
    'month': 'first'
}).reset_index()

print(f"✅ Aggregated data shape: {daily_product_sales.shape}")
print(f"\n📊 Sample aggregated data:")
daily_product_sales.head()

In [ ]:
# Create features for prediction
# For each product, create rolling averages and lag features

ml_data = []

for product in daily_product_sales['product_name'].unique():
    product_df = daily_product_sales[daily_product_sales['product_name'] == product].copy()
    product_df = product_df.sort_values('date')

    # Calculate rolling averages (7-day and 14-day)
    product_df['qty_rolling_7'] = product_df['quantity'].rolling(window=7, min_periods=1).mean()
    product_df['qty_rolling_14'] = product_df['quantity'].rolling(window=14, min_periods=1).mean()

    # Lag features (previous day sales)
    product_df['qty_lag_1'] = product_df['quantity'].shift(1)
    product_df['qty_lag_7'] = product_df['quantity'].shift(7)

    ml_data.append(product_df)

ml_df = pd.concat(ml_data, ignore_index=True)

# Fill NaN values from lag features
ml_df = ml_df.fillna(0)

print(f"✅ ML dataset shape: {ml_df.shape}")
print(f"\n📊 Features created:")
print(ml_df.columns.tolist())

## 🎯 Step 7: Train Forecasting Model

In [ ]:
# Select features for training
feature_columns = [
    'day_of_week',
    'is_weekend',
    'week_of_year',
    'month',
    'qty_rolling_7',
    'qty_rolling_14',
    'qty_lag_1',
    'qty_lag_7'
]

target_column = 'quantity'

# Prepare X and y
X = ml_df[feature_columns]
y = ml_df[target_column]

print(f"✅ Features (X): {X.shape}")
print(f"✅ Target (y): {y.shape}")
print(f"\n📊 Feature columns:")
for col in feature_columns:
    print(f"   • {col}")

In [ ]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"✅ Data split completed:")
print(f"   Training set: {X_train.shape[0]:,} samples")
print(f"   Test set: {X_test.shape[0]:,} samples")
print(f"   Split ratio: 80% train / 20% test")

In [ ]:
# Train Linear Regression model
print("🎯 Training Linear Regression model...\n")

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Make predictions
y_pred_lr = lr_model.predict(X_test)

# Evaluate
mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
rmse_lr = np.sqrt(mse_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print("📊 Linear Regression Performance:")
print(f"   • MAE (Mean Absolute Error): {mae_lr:.2f}")
print(f"   • RMSE (Root Mean Squared Error): {rmse_lr:.2f}")
print(f"   • R² Score: {r2_lr:.4f}")
print(f"   • Accuracy: {max(0, r2_lr) * 100:.1f}%")

In [ ]:
# Train Random Forest model
print("\n🌲 Training Random Forest model...\n")

rf_model = RandomForestRegressor(
    n_estimators=100,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

# Make predictions
y_pred_rf = rf_model.predict(X_test)

# Evaluate
mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
rmse_rf = np.sqrt(mse_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print("📊 Random Forest Performance:")
print(f"   • MAE (Mean Absolute Error): {mae_rf:.2f}")
print(f"   • RMSE (Root Mean Squared Error): {rmse_rf:.2f}")
print(f"   • R² Score: {r2_rf:.4f}")
print(f"   • Accuracy: {max(0, r2_rf) * 100:.1f}%")

In [ ]:
# Compare models
print("\n" + "=" * 60)
print("📊 MODEL COMPARISON")
print("=" * 60)

comparison = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'MAE': [mae_lr, mae_rf],
    'RMSE': [rmse_lr, rmse_rf],
    'R² Score': [r2_lr, r2_rf],
    'Accuracy %': [max(0, r2_lr) * 100, max(0, r2_rf) * 100]
})

print(comparison.to_string(index=False))

# Select best model
if r2_rf > r2_lr:
    best_model = rf_model
    best_name = 'Random Forest'
    best_r2 = r2_rf
else:
    best_model = lr_model
    best_name = 'Linear Regression'
    best_r2 = r2_lr

print(f"\n🏆 Best Model: {best_name} (R² = {best_r2:.4f})")

## 📈 Step 8: Visualize Model Performance

In [ ]:
# Plot predictions vs actual
plt.figure(figsize=(12, 6))

# Sample 200 points for clearer visualization
sample_size = min(200, len(y_test))
sample_indices = np.random.choice(len(y_test), sample_size, replace=False)

plt.scatter(y_test.iloc[sample_indices], y_pred_rf[sample_indices], alpha=0.5, label='Predictions')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Quantity', fontsize=12)
plt.ylabel('Predicted Quantity', fontsize=12)
plt.title(f'🎯 Predicted vs Actual Sales ({best_name})', fontsize=16, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Feature importance (for Random Forest)
if best_name == 'Random Forest':
    feature_importance = pd.DataFrame({
        'Feature': feature_columns,
        'Importance': rf_model.feature_importances_
    }).sort_values('Importance', ascending=False)

    plt.figure(figsize=(10, 6))
    plt.barh(feature_importance['Feature'], feature_importance['Importance'], color='teal')
    plt.xlabel('Importance', fontsize=12)
    plt.ylabel('Feature', fontsize=12)
    plt.title('📊 Feature Importance', fontsize=16, fontweight='bold')
    plt.gca().invert_yaxis()
    plt.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()

    print("\n🔍 Feature Importance:")
    print(feature_importance.to_string(index=False))

## 💾 Step 9: Save Trained Model

In [ ]:
# Prepare model package
model_package = {
    'model': best_model,
    'model_name': best_name,
    'feature_columns': feature_columns,
    'r2_score': best_r2,
    'trained_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'training_samples': len(X_train),
    'test_samples': len(X_test)
}

# Save to pickle file
model_filename = 'forecasting_model.pkl'

with open(model_filename, 'wb') as f:
    pickle.dump(model_package, f)

print(f"✅ Model saved: {model_filename}")
print(f"\n📦 Model Package Contents:")
print(f"   • Model: {best_name}")
print(f"   • R² Score: {best_r2:.4f}")
print(f"   • Features: {len(feature_columns)}")
print(f"   • Training samples: {len(X_train):,}")
print(f"   • Trained: {model_package['trained_date']}")

In [ ]:
# Download model file
from google.colab import files

print("⬇️ Downloading model file...\n")
files.download(model_filename)

print("\n✅ Download complete!")
print("\n📋 Next Steps:")
print("   1. Upload forecasting_model.pkl to your Django project")
print("   2. Run: python integrate_ml_model.py")
print("   3. Your Django app will use this model for predictions")
print("\n🛡️ Database Safety:")
print("   ✅ Your PostgreSQL database was NOT touched during training")
print("   ✅ Model will only READ from database when making predictions")
print("   ✅ Railway connections remain safe")

## 🎓 Step 10: Documentation for Thesis

### Model Summary:
- **Model Type:** {best_name}
- **R² Score:** {best_r2:.4f}
- **Training Data:** {len(X_train)} samples
- **Test Data:** {len(X_test)} samples
- **Features Used:** {len(feature_columns)}

### Key Findings:
1. The model successfully predicts daily sales with {max(0, best_r2) * 100:.1f}% accuracy
2. Historical patterns (rolling averages) are important predictors
3. Weekday/weekend patterns affect sales significantly

### Thesis Sections to Include:
1. **Methodology:** Explain your data collection and ML approach
2. **Data Description:** Describe your dataset (real + synthetic if used)
3. **Model Training:** Document the models you tested
4. **Results:** Present your R² scores and visualizations
5. **Implementation:** Show integration with Django system

### Ethical Considerations:
- ✅ Clearly label synthetic data if used
- ✅ Explain limitations of the model
- ✅ Discuss real-world applicability

---

**Good luck with your thesis! 🎓**
